# HW01-C — Airflow Scheduled Pipeline

A pipeline that only runs when you remember to click it is a chore.

Here you turn the SQL work into an Airflow DAG. The DAG refreshes your materialized view, validates it, and writes a run report.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- Airflow DAGs: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/dags.html
- Airflow Variables: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/variables.html
- Airflow best practices: https://airflow.apache.org/docs/apache-airflow/stable/best-practices.html

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [1]:
from pathlib import Path
import os, re, textwrap

PROJECT = Path.cwd()
for path in ['dags', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(exist_ok=True)

student_id = os.getenv('QBC12_STUDENT_ID', '8727') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', student_id.lower())
DAG_ID = f'qbc12_hw01_{safe_student}_airbnb_pipeline'
STUDENT_SCHEMA = os.getenv('QBC12_STUDENT_SCHEMA', 'student_ali_ghanbari')
DAG_ID, STUDENT_SCHEMA


('qbc12_hw01_8727_airbnb_pipeline', 'student_ali_ghanbari')

## 1. DAG design

Build this chain:

```text
read_config → refresh_summary → validate_summary → branch → success/failure report
```

Database credentials must come from Airflow Variables.

In [2]:
# TODO 1.1 - COMPLETED
# Create dags/<DAG_ID>.py.
# Include imports, DAG metadata, make_engine(), and a read_config task.
# Use Airflow Variables for DB credentials.

dag_source = 'from __future__ import annotations\n\nimport json\nfrom datetime import timedelta\nfrom pathlib import Path\nfrom urllib.parse import quote_plus\n\nimport pendulum\nfrom airflow.decorators import dag, task\nfrom airflow.models import Variable\nfrom sqlalchemy import create_engine, text\n\nSTUDENT_SCHEMA = "student_ali_ghanbari"\nMATERIALIZED_VIEW = "mv_airbnb_neighbourhood_summary"\nAIRFLOW_URL = "http://185.50.38.163:33013"\n\n\ndef make_engine(config: dict):\n    password = quote_plus(config["db_password"])\n    user = quote_plus(config["db_user"])\n    url = (\n        f"postgresql+psycopg2://{user}:{password}@"\n        f"{config[\'db_host\']}:{config[\'db_port\']}/{config[\'db_name\']}"\n    )\n    return create_engine(url, pool_pre_ping=True)\n\n\n@dag(\n    dag_id="qbc12_hw01_8727_airbnb_pipeline",\n    description="Refresh and validate the HW01 Airbnb neighbourhood summary materialized view.",\n    start_date=pendulum.datetime(2026, 7, 5, tz="UTC"),\n    schedule="0 6 * * *",\n    catchup=False,\n    max_active_runs=1,\n    default_args={"owner": "ali_ghanbari", "retries": 1, "retry_delay": timedelta(minutes=2)},\n    tags=["qbc12", "hw01", "airbnb", "8727"],\n)\ndef airbnb_pipeline():\n    @task\n    def read_config() -> dict:\n        return {\n            "db_host": Variable.get("QBC12_DB_HOST", default_var="185.50.38.163"),\n            "db_port": Variable.get("QBC12_DB_PORT", default_var="32112"),\n            "db_name": Variable.get("QBC12_DB_NAME", default_var="qbc12_airbnb"),\n            "db_user": Variable.get("QBC12_DB_USER"),\n            "db_password": Variable.get("QBC12_DB_PASSWORD"),\n            "student_schema": Variable.get("QBC12_STUDENT_SCHEMA", default_var=STUDENT_SCHEMA),\n            "report_dir": Variable.get("QBC12_REPORT_DIR", default_var="/opt/airflow/reports"),\n            "airflow_url": AIRFLOW_URL,\n        }\n\n    @task\n    def refresh_summary(config: dict) -> dict:\n        schema = config["student_schema"]\n        refresh_sql = f\'\'\'\n        CREATE SCHEMA IF NOT EXISTS "{schema}";\n\n        DROP MATERIALIZED VIEW IF EXISTS "{schema}".{MATERIALIZED_VIEW};\n\n        CREATE MATERIALIZED VIEW "{schema}".{MATERIALIZED_VIEW} AS\n        with date_bounds as (\n            select min(date) as start_date\n            from core.calendar_day\n        ),\n        calendar_30 as (\n            select\n                cd.listing_id,\n                avg(cd.price) filter (where cd.price is not null) as avg_calendar_price_30,\n                avg(case when cd.available then 1.0 else 0.0 end) as availability_30_rate\n            from core.calendar_day cd\n            cross join date_bounds db\n            where cd.date >= db.start_date\n              and cd.date < db.start_date + interval \'30 days\'\n            group by cd.listing_id\n        ),\n        calendar_365 as (\n            select\n                listing_id,\n                avg(case when available then 1.0 else 0.0 end) as availability_365_rate\n            from core.calendar_day\n            group by listing_id\n        ),\n        review_counts as (\n            select\n                listing_id,\n                count(*)::bigint as total_reviews\n            from core.review\n            group by listing_id\n        )\n        select\n            n.name as neighbourhood,\n            count(*)::bigint as num_listings,\n            round(avg(l.listing_price)::numeric, 2) as avg_price,\n            round(percentile_cont(0.5) within group (order by l.listing_price)::numeric, 2) as median_price,\n            round(avg(l.minimum_nights)::numeric, 2) as avg_minimum_nights,\n            coalesce(sum(rc.total_reviews), 0)::bigint as total_reviews,\n            round(avg(coalesce(rc.total_reviews, 0))::numeric, 2) as reviews_per_listing,\n            round(avg(c30.availability_30_rate)::numeric, 4) as availability_30_rate,\n            round(avg(c365.availability_365_rate)::numeric, 4) as availability_365_rate\n        from core.listing l\n        join core.neighbourhood n on n.neighbourhood_id = l.neighbourhood_id\n        left join calendar_30 c30 on c30.listing_id = l.listing_id\n        left join calendar_365 c365 on c365.listing_id = l.listing_id\n        left join review_counts rc on rc.listing_id = l.listing_id\n        where l.listing_price is not null\n          and l.listing_price > 0\n        group by n.name;\n\n        CREATE UNIQUE INDEX IF NOT EXISTS idx_mv_airbnb_neighbourhood_summary_neighbourhood\n        ON "{schema}".{MATERIALIZED_VIEW} (neighbourhood);\n\n        CREATE INDEX IF NOT EXISTS idx_mv_airbnb_neighbourhood_summary_num_listings\n        ON "{schema}".{MATERIALIZED_VIEW} (num_listings DESC);\n\n        CREATE INDEX IF NOT EXISTS idx_mv_airbnb_neighbourhood_summary_avg_price\n        ON "{schema}".{MATERIALIZED_VIEW} (avg_price);\n        \'\'\'\n        statements = [stmt.strip() for stmt in refresh_sql.split(";") if stmt.strip()]\n        engine = make_engine(config)\n        with engine.begin() as conn:\n            for stmt in statements:\n                conn.execute(text(stmt))\n            row_count = conn.execute(text(f\'select count(*) from "{schema}".{MATERIALIZED_VIEW}\')).scalar()\n        return {"schema": schema, "object": f"{schema}.{MATERIALIZED_VIEW}", "row_count": int(row_count)}\n\n    @task\n    def validate_summary(config: dict) -> dict:\n        schema = config["student_schema"]\n        validation_sql = f\'\'\'\n        select\n            count(*)::int as row_count,\n            count(*) filter (where neighbourhood is null or neighbourhood = \'\')::int as null_neighbourhoods,\n            count(*) filter (where avg_price is null or median_price is null or avg_price <= 0 or median_price <= 0)::int as bad_prices,\n            count(*) filter (\n                where availability_30_rate is null\n                   or availability_365_rate is null\n                   or availability_30_rate < 0\n                   or availability_30_rate > 1\n                   or availability_365_rate < 0\n                   or availability_365_rate > 1\n            )::int as bad_availability\n        from "{schema}".{MATERIALIZED_VIEW}\n        \'\'\'\n        engine = make_engine(config)\n        with engine.begin() as conn:\n            result = conn.execute(text(validation_sql)).mappings().one()\n        checks = dict(result)\n        checks["passed"] = (\n            checks["row_count"] > 0\n            and checks["null_neighbourhoods"] == 0\n            and checks["bad_prices"] == 0\n            and checks["bad_availability"] == 0\n        )\n        return checks\n\n    @task.branch\n    def choose_report_path(validation: dict) -> str:\n        return "write_success_report" if validation["passed"] else "write_failure_report"\n\n    def write_report(config: dict, refresh: dict, validation: dict, status: str) -> str:\n        report_dir = Path(config["report_dir"])\n        report_dir.mkdir(parents=True, exist_ok=True)\n        path = report_dir / "hw01_c_airflow_run_report.md"\n        path.write_text(\n            "# HW01-C Airflow Run Report\\n\\n"\n            f"- Status: {status}\\n"\n            f"- Airflow URL: {config[\'airflow_url\']}\\n"\n            f"- Refreshed object: `{refresh[\'object\']}`\\n"\n            f"- Row count: {refresh[\'row_count\']}\\n"\n            f"- Validation: `{json.dumps(validation, sort_keys=True)}`\\n"\n        )\n        return str(path)\n\n    @task\n    def write_success_report(validation: dict, refresh: dict, config: dict) -> str:\n        return write_report(config, refresh, validation, "success")\n\n    @task\n    def write_failure_report(validation: dict, refresh: dict, config: dict) -> str:\n        path = write_report(config, refresh, validation, "failure")\n        raise ValueError(f"Validation failed; report written to {path}")\n\n    config = read_config()\n    refresh = refresh_summary(config)\n    validation = validate_summary(config)\n    refresh >> validation\n    branch = choose_report_path(validation)\n    success = write_success_report(validation, refresh, config)\n    failure = write_failure_report(validation, refresh, config)\n    branch >> [success, failure]\n\n\nairbnb_pipeline()\n'
dag_path = PROJECT / 'dags' / f'{DAG_ID}.py'
dag_path.write_text(dag_source)
print(dag_path)


/Users/mahbod/Desktop/Quera/Mlops/deadlines/W1/3/dags/qbc12_hw01_8727_airbnb_pipeline.py


## 2. Refresh task

The refresh task should recreate your materialized view in Postgres. Do not move the full dataset into Python.

In [3]:
# TODO 2.1 - COMPLETED
# Add refresh_summary(config).
# It should create/refresh your materialized view and indexes.
# Return a small dict, not a dataframe.

# Implemented in the generated DAG file above as refresh_summary(config).
print((PROJECT / 'dags' / f'{DAG_ID}.py').read_text().split('def refresh_summary(config: dict) -> dict:', 1)[1][:1600])



        schema = config["student_schema"]
        refresh_sql = f'''
        CREATE SCHEMA IF NOT EXISTS "{schema}";

        DROP MATERIALIZED VIEW IF EXISTS "{schema}".{MATERIALIZED_VIEW};

        CREATE MATERIALIZED VIEW "{schema}".{MATERIALIZED_VIEW} AS
        with date_bounds as (
            select min(date) as start_date
            from core.calendar_day
        ),
        calendar_30 as (
            select
                cd.listing_id,
                avg(cd.price) filter (where cd.price is not null) as avg_calendar_price_30,
                avg(case when cd.available then 1.0 else 0.0 end) as availability_30_rate
            from core.calendar_day cd
            cross join date_bounds db
            where cd.date >= db.start_date
              and cd.date < db.start_date + interval '30 days'
            group by cd.listing_id
        ),
        calendar_365 as (
            select
                listing_id,
                avg(case when available then 1.0 else 0.0 end) 

## 3. Validation task

Required checks:

- row_count > 0
- null_neighbourhoods == 0
- bad_prices == 0
- bad_availability == 0

In [4]:
# TODO 3.1 - COMPLETED
# Add validate_summary(config).
# Return a dict containing each check and passed=True/False.

# Implemented in the generated DAG file above as validate_summary(config).
print((PROJECT / 'dags' / f'{DAG_ID}.py').read_text().split('def validate_summary(config: dict) -> dict:', 1)[1][:1200])



        schema = config["student_schema"]
        validation_sql = f'''
        select
            count(*)::int as row_count,
            count(*) filter (where neighbourhood is null or neighbourhood = '')::int as null_neighbourhoods,
            count(*) filter (where avg_price is null or median_price is null or avg_price <= 0 or median_price <= 0)::int as bad_prices,
            count(*) filter (
                where availability_30_rate is null
                   or availability_365_rate is null
                   or availability_30_rate < 0
                   or availability_30_rate > 1
                   or availability_365_rate < 0
                   or availability_365_rate > 1
            )::int as bad_availability
        from "{schema}".{MATERIALIZED_VIEW}
        '''
        engine = make_engine(config)
        with engine.begin() as conn:
            result = conn.execute(text(validation_sql)).mappings().one()
        checks = dict(result)
        checks["passed"] = (
  

## 4. Branching and reports

Success and failure should not look the same.

Use `@task.branch` to choose the report path.

In [5]:
# TODO 4.1 - COMPLETED
# Add choose_report_path, write_success_report, and write_failure_report.
# Failure report should raise ValueError after writing the report.

# Implemented in the generated DAG file above using @task.branch.
print((PROJECT / 'dags' / f'{DAG_ID}.py').read_text().split('def choose_report_path(validation: dict) -> str:', 1)[1][:1400])



        return "write_success_report" if validation["passed"] else "write_failure_report"

    def write_report(config: dict, refresh: dict, validation: dict, status: str) -> str:
        report_dir = Path(config["report_dir"])
        report_dir.mkdir(parents=True, exist_ok=True)
        path = report_dir / "hw01_c_airflow_run_report.md"
        path.write_text(
            "# HW01-C Airflow Run Report\n\n"
            f"- Status: {status}\n"
            f"- Airflow URL: {config['airflow_url']}\n"
            f"- Refreshed object: `{refresh['object']}`\n"
            f"- Row count: {refresh['row_count']}\n"
            f"- Validation: `{json.dumps(validation, sort_keys=True)}`\n"
        )
        return str(path)

    @task
    def write_success_report(validation: dict, refresh: dict, config: dict) -> str:
        return write_report(config, refresh, validation, "success")

    @task
    def write_failure_report(validation: dict, refresh: dict, config: dict) -> str:
        path = w

In [6]:
# Syntax check. This is not a full Airflow run.
import py_compile

dag_path = PROJECT / 'dags' / f'{DAG_ID}.py'
assert dag_path.exists(), f'Missing DAG file: {dag_path}'
py_compile.compile(str(dag_path), doraise=True)
print('DAG compiles:', dag_path)

DAG compiles: /Users/mahbod/Desktop/Quera/Mlops/deadlines/W1/3/dags/qbc12_hw01_8727_airbnb_pipeline.py


## 5. Shared Airflow run

In shared Airflow:

1. find your DAG
2. unpause it
3. trigger it manually
4. inspect Graph view
5. inspect logs
6. confirm the materialized view was refreshed

Screenshots:

```text
screenshots/airflow_dag_graph.png
screenshots/airflow_success_run.png
```

In [7]:
# TODO 5.1 - COMPLETED
# Write reports/hw01_c_airflow.md.
# Include DAG id, Airflow URL, successful run timestamp,
# refreshed object name, validation result, screenshot paths.

report = f'''# HW01-C Airflow Scheduled Pipeline

- DAG id: `{DAG_ID}`
- Airflow URL: http://185.50.38.163:33013
- Schedule: daily at 06:00 UTC
- Refreshed object: `{STUDENT_SCHEMA}.mv_airbnb_neighbourhood_summary`
- Validation checks: row_count > 0, null_neighbourhoods == 0, bad_prices == 0, bad_availability == 0
- Local syntax check: passed by `py_compile`
- Shared Airflow run timestamp: pending manual/shared-Airflow trigger
- Graph screenshot path: `screenshots/airflow_dag_graph.png`
- Successful run screenshot path: `screenshots/airflow_success_run.png`

The DAG reads database credentials from Airflow Variables: `QBC12_DB_HOST`, `QBC12_DB_PORT`, `QBC12_DB_NAME`, `QBC12_DB_USER`, and `QBC12_DB_PASSWORD`. It recreates the materialized view in Postgres, validates the resulting summary, then branches to a success or failure report task.
'''
Path('reports/hw01_c_airflow.md').write_text(report)
print(report)


# HW01-C Airflow Scheduled Pipeline

- DAG id: `qbc12_hw01_8727_airbnb_pipeline`
- Airflow URL: http://185.50.38.163:33013
- Schedule: daily at 06:00 UTC
- Refreshed object: `student_ali_ghanbari.mv_airbnb_neighbourhood_summary`
- Validation checks: row_count > 0, null_neighbourhoods == 0, bad_prices == 0, bad_availability == 0
- Local syntax check: passed by `py_compile`
- Shared Airflow run timestamp: pending manual/shared-Airflow trigger
- Graph screenshot path: `screenshots/airflow_dag_graph.png`
- Successful run screenshot path: `screenshots/airflow_success_run.png`

The DAG reads database credentials from Airflow Variables: `QBC12_DB_HOST`, `QBC12_DB_PORT`, `QBC12_DB_NAME`, `QBC12_DB_USER`, and `QBC12_DB_PASSWORD`. It recreates the materialized view in Postgres, validates the resulting summary, then branches to a success or failure report task.



In [8]:
for file in [f'dags/{DAG_ID}.py', 'reports/hw01_c_airflow.md']:
    assert Path(file).exists(), f'Missing {file}'
print('Basic deliverables exist.')

Basic deliverables exist.


## Commit

```bash
git add dags reports screenshots notebooks
git commit -m "HW01-C Airflow scheduled pipeline"
```